# End-to-End SENSEX Analysis
This notebook fulfills all 6 objectives on the provided `BSE SENSEX.csv`.

## 1️⃣ Objective: Data Loading and Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the 'BSE SENSEX.csv' file is uploaded to your Colab environment
# You can use the file sidebar in Colab to upload the file before running this cell.
try:
    df = pd.read_csv('BSE SENSEX.csv')
except FileNotFoundError:
    print('Please upload BSE SENSEX.csv using the left sidebar in Google Colab.')
    raise

display(df.head())
print('\n--- Dataset Info ---')
df.info()
print('\n--- Summary Statistics ---')
display(df.describe())

## 2️⃣ Objective: Data Preprocessing and Cleaning

In [ ]:
print('Missing values:')
display(df.isnull().sum())

# Handling missing values (Forward fill)
df.ffill(inplace=True)

# Data transformation (Convert Date to datetime format and sort chronologically)
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)

print('\nData after preprocessing:')
display(df.head())
print('\nMissing values after cleaning:')
display(df.isnull().sum())

## 3️⃣ Objective: Exploratory Data Analysis (EDA)

In [ ]:
# Set style for plots
sns.set_theme(style='whitegrid')

plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'], color='blue', label='SENSEX Close Price')
plt.title('SENSEX Closing Price Over Time', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Closing Price (INR)')
plt.legend()
plt.show()

# Daily Returns Distribution (Statistical Analysis)
df['Daily_Return'] = df['Close'].pct_change() * 100

plt.figure(figsize=(10, 5))
sns.histplot(df['Daily_Return'].dropna(), bins=50, kde=True, color='green')
plt.title('Distribution of Daily Returns %', fontsize=16)
plt.xlabel('Daily Return (%)')
plt.show()

print('\nStatistical analysis on Daily Returns:')
display(df['Daily_Return'].describe())

## 4️⃣ Objective: Identify Relationships Between Features

In [ ]:
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Daily_Return']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Features', fontsize=16)
plt.show()

# Pairplot of selected features over a random sample to maintain fast execution
sample_df = df[['Open', 'High', 'Low', 'Close']].dropna().sample(n=min(500, len(df)), random_state=42)
sns.pairplot(sample_df, corner=True, diag_kind='kde')
plt.suptitle('Pairplot of Main Features', y=1.02)
plt.show()

## 5️⃣ Objective: Dimensionality Reduction using PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Features for PCA
features = ['Open', 'High', 'Low']
X_pca_input = df[features].dropna()

# Standardizing the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca_input)

# Applying PCA
pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])

print('Explained Variance Ratio by the Components:', pca.explained_variance_ratio_)

plt.figure(figsize=(8, 6))
plt.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.5, c='purple')
plt.title('PCA of SENSEX Features')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

## 6️⃣ Objective: Build a Prediction Model (Linear Regression)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df_model = df.dropna().copy()
X = df_model[['Open', 'High', 'Low']]
y = df_model['Close']

# Time-series split (no shuffle)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# Model Building and Training
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('--- Model Evaluation Metrics ---')
print(f'Root Mean Squared Error (RMSE): {rmse:.2f}')
print(f'Mean Absolute Error (MAE): {mae:.2f}')
print(f'R-squared Score (R2): {r2:.4f}')

# Visualizing Results
plt.figure(figsize=(14, 6))
test_dates = df_model.loc[X_test.index, 'Date']

plt.plot(test_dates, y_test, label='Actual Close Price', color='blue')
plt.plot(test_dates, y_pred, label='Predicted Close Price', color='red', linestyle='--')
plt.title('Actual vs Predicted SENSEX Closing Price', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.legend()
plt.show()